In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.common.action_chains import ActionChains
    

In [222]:


# Explicit Wait를 활용해서 스크래핑이 잘 이루어지도록 코드를 작성해봅시다.
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.execute_script('window.open("about:blank", "_blank");')
driver.execute_script('window.open("about:blank", "_blank");')
tabs = driver.window_handles
driver.switch_to.window(tabs[0])


driver.get("https://www.wanted.co.kr/wdlist/518/899?country=kr&job_sort=job.popularity_order&years=-1&selected=899&locations=all")
WebDriverWait(driver, 10)



<selenium.webdriver.support.wait.WebDriverWait (session="08195161c17d06e12d41c141a5cca022")>

In [ ]:





prev_height = driver.execute_script("return document. body.scrollHeight")
import time
while True:
    driver.execute_script("window.scrollTo(0,document.body.scrollHeight)")
    time.sleep(2)
    current_height = driver.execute_script("return document. body.scrollHeight")
    print(prev_height,current_height)
    if current_height == prev_height:
        break
    prev_height = current_height


In [ ]:
driver.switch_to.window(tabs[0])

In [ ]:
cnt = 1
while True:
    cnt+=1
    try:
        notice = driver.find_element(By.XPATH, f'//*[@id="__next"]/div[3]/div[2]/ul/li[{cnt}]/div/a')
        notice_url = notice.get_attribute('href')
        print(notice_url)
        driver.switch_to.window(tabs[1])
        driver.get(notice_url)
        WebDriverWait(driver, 10)
        driver.find_element(By.XPATH, '//*[@id="__next"]/main/div[1]/div/section/section/article[1]/div/button/span[2]').click()


        
        driver.switch_to.window(tabs[0])
        time.sleep(1)
    except NoSuchElementException:
        break

In [ ]:
company_url_set = set()
key = { 
    '공고PK':'notice_id',
    '회사PK':'company_id',
    '직무분야':'notice_job_category',
    '근무지역':'notice_location',
    '경력사항':'notice_career',
    '공고제목':'notice_title',
    '포지션상세':'notice_position',
    '주요업무':'notice_main_work',
    '자격요건':'notice_qualification',
    '우대사항':'notice_preferred_qualification',
    '혜택 및 복지':'notice_welfare',
    '채용 전형':'notice_category',
    '마감일':'notice_end_date',
    '기술 스택 • 툴':'notice_tech_stack',
    '공고URL':'notice_url',
    '회사명':'company_name',
    '태그':'company_tag',
    '연봉':'company_salary',
    '위치':'company_location',
    '인원':'company_headcount',
    '매출':'company_revenue',
    '기업 정보':'company_info',
    '등록일시':'reg_dt',
    '수정일시':'mod_dt'
}



In [247]:
driver.switch_to.window(tabs[0])
notice = driver.find_element(By.XPATH, f'//*[@id="__next"]/div[3]/div[2]/ul/li[{cnt}]/div/a')
notice_url = notice.get_attribute('href')
driver.switch_to.window(tabs[1])
driver.get(notice_url)
WebDriverWait(driver, 10)
#상세보기 클릭
driver.find_element(By.XPATH, '//*[@id="__next"]/main/div[1]/div/section/section/article[1]/div/button/span[2]').click()
row_data = {}
row_data[key['공고URL']] = notice_url
row_data[key['공고PK']] = notice_url.split('/')[-1]
row_data[key['직무분야']] = '파이썬 개발자'
row_data[key['공고제목']] = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div[1]/div/section/header/h1').text
row_data[key['경력사항']] = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div[1]/div/section/header/div/div[1]/span[4]').text
notice_context = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div[1]/div/section/section').find_elements(By.TAG_NAME,'article')
#포지션 상세~ 채용전형 수집
notice_description = notice_context[0]
row_data[key['포지션상세']] = notice_description.find_element(By.TAG_NAME, 'div').find_element(By.TAG_NAME, 'span').text
for div in notice_description.find_element(By.TAG_NAME, 'div').find_elements(By.TAG_NAME, 'div'):
    context_title = div.find_element(By.TAG_NAME, 'h3').text.strip()
    row_data[key[context_title]] = div.find_element(By.TAG_NAME, 'span').text

#기술스택~근무지역
for notice_context_other in notice_context[1:]:
    #print(notice_context_other.text,'\n')
    context_title = notice_context_other.find_element(By.TAG_NAME, 'h2').text.strip()
    print(context_title)
    if context_title == '기술 스택 • 툴':
        row_data[key[context_title]] = notice_context_other.find_element(By.TAG_NAME, 'ul').text
    elif context_title == '마감일':
        row_data[key[context_title]] = notice_context_other.find_element(By.TAG_NAME, 'span').text
    elif context_title == '근무지역':
        row_data[key[context_title]] = notice_context_other.find_elements(By.TAG_NAME, 'div')[-1].text       
        break

#회사url 별도 처리를 위해 따로 변수에 저장
company_url = driver.find_element(By.XPATH, '//*[@id="__next"]/main/div[1]/div/section/header/div/div[1]/a').get_attribute('href')
company_url_set.add(company_url)
row_data[key['회사PK']] = company_url.split('/')[-1]
driver.switch_to.window(tabs[0])
time.sleep(1)

기술 스택 • 툴
태그
마감일
근무지역


In [ ]:
for k,v in row_data.items():
    print(k)


print(row_data[key['공고PK']])
print(row_data[key['회사PK']])
print(row_data[key['직무분야']])
print(row_data[key['근무지역']])
print(row_data[key['경력사항']])
print(row_data[key['공고제목']])
print(row_data[key['포지션상세']])
print(row_data[key['주요업무']])
print(row_data[key['자격요건']])
print(row_data[key['우대사항']])
print(row_data[key['혜택 및 복지']])
print(row_data[key['채용 전형']])
print(row_data[key['마감일']])
print(row_data[key['기술 스택 • 툴']])
print(row_data[key['공고URL']])
print(row_data[key['등록일시']])
print(row_data[key['수정일시']])

In [297]:
company_url_set = {"https://www.wanted.co.kr/company/47104"}

for company_url in company_url_set:
    row_data = {}

    print(company_url)
    driver.switch_to.window(tabs[1])
    driver.get(company_url)
    WebDriverWait(driver, 10)

    while True:
        driver.execute_script("window.scrollTo(0,document.body.scrollHeight)")
        time.sleep(2)
        current_height = driver.execute_script("return document. body.scrollHeight")
        print(prev_height,current_height)
        if current_height == prev_height:
            break
        prev_height = current_height

    row_data[key['회사명']] = driver.find_element(By.XPATH, '//*[@id="__next"]/div[3]/div[2]/div/div[1]/div[1]/div[1]/h1').text
    company_detail_context = driver.find_element(By.XPATH, '//*[@id="__next"]/div[3]/div[2]/div/div[2]').find_elements(By.TAG_NAME,'section')
    for context in company_detail_context:
        context_title = context.find_element(By.TAG_NAME, 'h2').text.strip()
        if context_title == '태그':
            row_data[key[context_title]] = context.find_element(By.XPATH, './/div').text
        elif context_title == '연봉':
            row_data[key[context_title]] = context.find_element(By.XPATH, './/div[1]/div[2]/div[1]/div/div').text.replace('만원','').replace(',','')
        elif context_title == '매출':
            row_data[key[context_title]] = context.find_element(By.XPATH, './/div/div[1]/div/div').text
        elif context_title == '인원':
            row_data[key[context_title]] = context.find_element(By.XPATH, './/div/div[1]/div[1]/div/div').text.replace('명','').replace(',','')
        elif context_title == '기업 정보':
            pass
            # company_info = {}
            # info_sections = context.find_elements(By.TAG_NAME, 'dl')
            # for section in info_sections:
            #     title = section.find_element(By.TAG_NAME, 'dt').text.strip()
            #     value = section.find_element(By.TAG_NAME, 'dd').text.strip()
            #     company_info[title] = value
            # row_data[key['기업 정보']] = company_info
    
    location_element = driver.find_element(By.XPATH, '//*[@id="__next"]/div[3]/div[2]/div/div[2]/div[1]')
    ActionChains(driver).scroll_to_element(location_element).perform()
    WebDriverWait(location_element.find_element(By.TAG_NAME, 'span'), 10)
    row_data[key['위치']] = location_element.find_elements(By.TAG_NAME, 'span')[-1].text


    print(row_data)


https://www.wanted.co.kr/company/47104
5210 4947
4947 4947
{'company_name': '두어스', 'company_tag': '50명이하\n인원 급성장\n연봉상위6~10%\n건강검진지원\n식대지원\n자기계발지원\n장비지원\n스톡옵션\n커피·스낵바', 'company_salary': '4416', 'company_headcount': '17', 'company_location': '서울 강남구 역삼동 668-16, 오피스B 6층, 두어스'}


In [278]:
row_data[key['위치']].text
    

'위치\n© NAVER Corp.\n경기 성남시 분당구 분당내곡로 131, 판교테크원타워 타워2 15층'